# Exp36R — PRR-Stat restricted ablation on the frozen Exp33 protocol

This notebook refits **PRR-Stat**, the restricted fixed-Pearson-support ablation, and compares it with the main PRR method on the 24 PatchTST dataset–horizon cells.

## What is compared

\[
\boxed{\text{Pearson baseline} \quad \text{vs.} \quad \text{PRR-Stat (ablation)} \quad \text{vs.} \quad \text{PRR (main)}}
\]

**PRR-Stat** keeps:

- Pearson Top-100 candidate support only;
- seven handcrafted past-observable statistics;
- a 29-D pair feature \(1+4\times 7\);
- the small residual MLP and future-supervised listwise target;
- Top-10 historical futures.

**PRR** additionally expands support with learned-representation Top-100 retrieval and uses the 285-D statistical + representation fusion reranker.

The ablation isolates future-supervised reranking within fixed Pearson support. Full PRR then adds both learned embedding relations and learned candidate-support expansion, so the PRR-vs.-PRR-Stat increment is a combined representation-and-support effect.

No forecasting backbone is trained or executed here.

## PRR-Stat optimization recipe

- AdamW, learning rate 1e-3, weight decay 1e-4;
- batch size 128 queries;
- maximum 30 epochs, patience 6;
- gradient clipping 5.0;
- future-target temperature 0.5;
- residual scale initialized at 0.1;
- chronological epoch selection, followed by reinitialization and refit using historical supervision available before test.

**Aligned protocol note (final paper):** all six datasets use `memory_stride=8` for the long-horizon Pearson / PRR-Stat / full-PRR comparison. Stored outputs were cleared after the stride-alignment audit to avoid displaying stale stride-24 values.

In [ ]:

# Cell 1 — configuration

from pathlib import Path
import os, json, math, random, time, gc, hashlib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

ROOT = Path(os.environ.get("PRR_ROOT", str(Path.cwd().resolve().parents[1] if Path.cwd().name == "prr_long_horizon" else Path.cwd().resolve())))
EXP33 = ROOT / "results" / "exp33"
EXP34 = ROOT / "results" / "exp34"
EXP35 = ROOT / "results" / "exp35B"

MANIFEST_ROOT = EXP33 / "final_eval_manifest"
DIRECT_ROOT = EXP34 / "patchtst_direct_cache"

RESULT_ROOT = ROOT / "results" / "exp36R"
CKPT_ROOT = RESULT_ROOT / "prr_stat_checkpoints"
CACHE_ROOT = RESULT_ROOT / "prr_stat_retrieval_cache"

RESULT_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = ("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS = (96,192,336,720)

CHANNELS = {
    "ETTh1":7,
    "Weather":21,
    "Electricity":321,
    "Traffic":862,
    "Exchange":8,
    "Solar":137,
}

DEVICE = torch.device("cuda:0")
assert torch.cuda.is_available()

class CFG:
    retrieval_seq_len = 96
    top_m = 100
    top_k = 10

    # PRR-Stat optimization.
    lr = 1e-3
    wd = 1e-4
    batch = 128
    max_epochs = 30
    patience = 6
    grad_clip = 5.0
    target_tau = 0.5
    feature_clip = 8.0

    # Historical protocol.
    initial_memory_fraction = 0.60

    # Query caps from the paper; deterministic and channel-balanced.
    train_cap_mechanism = 6000       # ETTh1 / Weather
    train_cap_frozen = 10000         # Electricity/Traffic/Exchange/Solar
    seed = 0

cfg = CFG()
EPS = 1e-8

print("GPU:",torch.cuda.get_device_name(0))
print("Result:",RESULT_ROOT)


In [ ]:

# Cell 2 — exact PRR-Stat architecture

def set_seed(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))

def inverse_softplus(x):
    return math.log(math.exp(float(x))-1.0)

class PRRStatReranker(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(29,128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128,128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128,1),
        )
        self.raw_alpha = nn.Parameter(
            torch.tensor(
                inverse_softplus(0.1),
                dtype=torch.float32,
            )
        )

    def forward(self,feat,base):
        delta=self.net(feat).squeeze(-1)
        alpha=F.softplus(self.raw_alpha)
        return base+alpha*delta

print("PRR-Stat input dim =",29)
print("Parameters =",sum(p.numel() for p in PRRStatReranker().parameters()))


In [ ]:

# Cell 3 — datasets, train-only normalization, protocol boundaries

DATA_CANDIDATES = {
    "ETTh1":[
        Path("/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv"),
        Path("/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv"),
    ],
    "Weather":[
        Path("/data/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather/weather.csv"),
    ],
    "Electricity":[
        Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
    ],
    "Traffic":[
        Path("/data/Time-Series-Library/dataset/traffic/traffic.csv"),
        Path("/data/Time-Series-Library_v2/dataset/traffic/traffic.csv"),
    ],
    "Exchange":[
        Path("/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv"),
    ],
    "Solar":[
        Path("/data/Time-Series-Library/dataset/Solar/solar_AL.txt"),
        Path("/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt"),
    ],
}

def read_raw(dataset):
    p=next((p for p in DATA_CANDIDATES[dataset] if p.exists()),None)
    assert p is not None,dataset

    if dataset=="Solar":
        try:
            df=pd.read_csv(p,header=None)
            if df.shape[1]==1:
                df=pd.read_csv(p,header=None,sep=r"\s+")
        except Exception:
            df=pd.read_csv(p,header=None,sep=r"\s+")
    else:
        df=pd.read_csv(p)
        if "date" in df.columns:
            df=df.drop(columns=["date"])
        if "OT" in df.columns:
            df=df[[x for x in df.columns if x!="OT"]+["OT"]]

    df=(
        df.apply(pd.to_numeric,errors="coerce")
        .dropna(axis=1,how="all")
        .replace([np.inf,-np.inf],np.nan)
        .interpolate(axis=0,limit_direction="both")
        .ffill()
        .bfill()
    )
    return df.to_numpy(np.float32)

def split_bounds(dataset,n):
    if dataset=="ETTh1":
        train_end=12*30*24
        val_end=train_end+4*30*24
    else:
        train_end=int(.70*n)
        val_end=n-int(.20*n)
    return train_end,val_end,n

def memory_stride(dataset):
    # Final aligned long-horizon protocol: common stride=8 for all six datasets.
    return 8

DATA={}

for dataset in DATASETS:
    raw=read_raw(dataset)
    assert raw.shape[1]==CHANNELS[dataset],(dataset,raw.shape)

    tr,va,te=split_bounds(dataset,len(raw))

    mu=raw[:tr].mean(axis=0).astype(np.float32)
    sd=raw[:tr].std(axis=0,ddof=0).astype(np.float32)
    assert np.all(sd>1e-6)

    z=((raw-mu[None,:])/sd[None,:]).astype(np.float32)

    DATA[dataset]=dict(
        z=z,
        train_end=tr,
        val_end=va,
        test_end=te,
    )

    print(dataset,raw.shape,"train",tr,"val",va)

print("[PASS] six datasets ready")


In [ ]:

# Cell 4 — observable features and deterministic pair sampling

def batch_pattern(x):
    x=np.asarray(x,np.float32)
    xc=x-x.mean(axis=-1,keepdims=True)
    n=np.linalg.norm(xc,axis=-1,keepdims=True)
    return np.where(
        n>EPS,
        xc/np.maximum(n,EPS),
        0.0,
    ).astype(np.float32)

def context7(x):
    x=np.asarray(x,np.float32)
    short=max(8,cfg.retrieval_seq_len//4)

    m=x.mean(axis=-1)
    s=x.std(axis=-1)+EPS

    f1=(x[...,-1]-m)/s
    f2=(x[...,-short:].mean(axis=-1)-m)/s
    f3=(x[...,-1]-x[...,-short])/s
    f4=(x[...,-1]-x[...,0])/s

    dx=np.diff(x,axis=-1)
    ds=np.diff(x[...,-short:],axis=-1)
    f5=(ds.std(axis=-1)+EPS)/(dx.std(axis=-1)+EPS)

    t=np.linspace(-1.,1.,cfg.retrieval_seq_len,dtype=np.float32)
    t=t-t.mean()
    f6=(
        np.sum(t*(x-m[...,None]),axis=-1)
        /(np.sum(t*t)+EPS)
    )/s

    a=x[...,:-1]
    b=x[...,1:]
    a=a-a.mean(axis=-1,keepdims=True)
    b=b-b.mean(axis=-1,keepdims=True)
    f7=np.sum(a*b,axis=-1)/(
        np.sqrt(
            np.sum(a*a,axis=-1)
            *np.sum(b*b,axis=-1)
        )+EPS
    )

    return np.stack(
        [f1,f2,f3,f4,f5,f6,f7],
        axis=-1,
    ).astype(np.float32)

def past_windows(z,ch,anchors):
    anchors=np.asarray(anchors,np.int64)
    idx=(
        anchors[:,None]
        -cfg.retrieval_seq_len
        +np.arange(cfg.retrieval_seq_len)[None,:]
    )
    return z[idx,ch].astype(np.float32)

def future_residuals(z,ch,anchors,h):
    anchors=np.asarray(anchors,np.int64)
    idx=anchors[:,None]+np.arange(h)[None,:]
    future=z[idx,ch].astype(np.float32)
    current=z[anchors-1,ch].astype(np.float32)
    return (future-current[:,None]).astype(np.float32)

def balanced_pairs(dataset,anchors,n_pairs,seed):
    anchors=np.asarray(anchors,np.int64)
    C=CHANNELS[dataset]

    n_pairs=min(int(n_pairs),len(anchors)*C)
    base=n_pairs//C
    rem=n_pairs%C

    rng=np.random.default_rng(seed)
    extras=set(
        rng.choice(C,size=rem,replace=False).tolist()
        if rem else []
    )

    aa=[]
    cc=[]

    for ch in range(C):
        n=base+(1 if ch in extras else 0)
        if n<=0:
            continue
        sel=rng.choice(
            anchors,
            size=n,
            replace=(n>len(anchors)),
        )
        aa.append(sel.astype(np.int64))
        cc.append(np.full(n,ch,np.int64))

    a=np.concatenate(aa)
    c=np.concatenate(cc)

    order=np.lexsort((c,a))
    return a[order],c[order]

print("[PASS] feature/sampling helpers ready")


In [ ]:

# Cell 5 — Exp33/Exp34 loaders

def load_manifest(dataset,h,split):
    p=MANIFEST_ROOT/f"{dataset}_H{h}_{split}.npz"
    assert p.exists(),p
    with np.load(p,allow_pickle=False) as z:
        return (
            np.asarray(z["anchor"],np.int64),
            np.asarray(z["channel"],np.int64),
        )

def load_direct(dataset,h,split):
    p=DIRECT_ROOT/f"{dataset}_H{h}_{split}_patchtst_direct.npz"
    assert p.exists(),p

    with np.load(p,allow_pickle=False) as z:
        a=np.asarray(z["anchor"],np.int64)
        c=np.asarray(z["channel"],np.int64)
        d=np.asarray(z["direct"],np.float32)
        y=np.asarray(z["truth"],np.float32)

    ma,mc=load_manifest(dataset,h,split)
    assert np.array_equal(a,ma)
    assert np.array_equal(c,mc)

    return a,c,d,y

for dataset in DATASETS:
    for h in HORIZONS:
        for split in ("val","test"):
            a,c,direct_arr,truth_arr=load_direct(dataset,h,split)

print("[PASS] 48 Exp34R caches align with Exp33")


In [ ]:

# Cell 6 — build PRR-Stat scaler from the initial training memory

def fit_feature_scaler(dataset,h):
    z=DATA[dataset]["z"]
    train_end=DATA[dataset]["train_end"]
    boundary=int(cfg.initial_memory_fraction*train_end)

    stride=memory_stride(dataset)

    anchors=np.arange(
        cfg.retrieval_seq_len,
        boundary-h+1,
        stride,
        dtype=np.int64,
    )

    all_stats=[]

    for ch in range(CHANNELS[dataset]):
        if len(anchors)==0:
            continue
        p=past_windows(z,ch,anchors)
        all_stats.append(context7(p))

    x=np.concatenate(all_stats,axis=0)
    med=np.median(x,axis=0).astype(np.float32)
    iqr=(
        np.percentile(x,75,axis=0)
        -np.percentile(x,25,axis=0)
    ).astype(np.float32)
    iqr=np.where(iqr<1e-5,1.0,iqr).astype(np.float32)

    return med,iqr

def scale_stats(x,med,iqr):
    return np.clip(
        (x-med)/iqr,
        -cfg.feature_clip,
        cfg.feature_clip,
    ).astype(np.float32)

print("[PASS] scaler helpers ready")


In [ ]:

# Cell 7 — construct listwise PRR-Stat supervision

def top_m_indices(scores,m):
    m=min(int(m),len(scores))
    part=np.argpartition(-scores,m-1)[:m]
    return part[np.argsort(-scores[part])]

def build_supervision(
    dataset,
    h,
    query_anchor,
    query_channel,
    memory_boundary,
    med,
    iqr,
    need_candidate_future=False,
):
    """
    Returns query-wise Pattern Top-M features and future-supervised targets.

    feat: [N,M,29]
    base: [N,M]
    target: [N,M]
    distance: [N,M]
    candidate_anchor: [N,M]
    """
    z=DATA[dataset]["z"]
    N=len(query_anchor)
    M=cfg.top_m

    feat=np.empty((N,M,29),np.float32)
    base=np.empty((N,M),np.float32)
    target=np.empty((N,M),np.float32)
    dist=np.empty((N,M),np.float32)
    cand_anchor=np.empty((N,M),np.int64)

    stride=memory_stride(dataset)

    for ch in np.unique(query_channel):
        rows=np.flatnonzero(query_channel==ch)
        qa=query_anchor[rows]

        mem_a=np.arange(
            cfg.retrieval_seq_len,
            int(memory_boundary)-h+1,
            stride,
            dtype=np.int64,
        )

        if len(mem_a)<M:
            raise ValueError(
                f"{dataset} H={h} ch={ch}: memory={len(mem_a)} < M={M}"
            )

        mem_past=past_windows(z,int(ch),mem_a)
        mem_pat=batch_pattern(mem_past)
        mem_stat=scale_stats(
            context7(mem_past),
            med,
            iqr,
        )
        mem_future=future_residuals(
            z,int(ch),mem_a,h
        )

        q_past=past_windows(z,int(ch),qa)
        q_pat=batch_pattern(q_past)
        q_stat=scale_stats(
            context7(q_past),
            med,
            iqr,
        )
        q_future=future_residuals(
            z,int(ch),qa,h
        )

        ps=q_pat@mem_pat.T

        for local,row in enumerate(rows):
            ix=top_m_indices(ps[local],M)

            b=ps[local,ix].astype(np.float32)
            cs=mem_stat[ix]
            qs=q_stat[local]

            qrep=np.repeat(
                qs[None,:],
                M,
                axis=0,
            )

            f=np.concatenate(
                [
                    b[:,None],
                    qrep,
                    cs,
                    qrep-cs,
                    np.abs(qrep-cs),
                ],
                axis=1,
            ).astype(np.float32)

            assert f.shape==(M,29)

            dd=np.mean(
                (
                    mem_future[ix]
                    -q_future[local][None,:]
                )**2,
                axis=1,
            ).astype(np.float32)

            mu=float(dd.mean())
            sd=float(dd.std())
            zdist=(dd-mu)/max(sd,1e-6)

            logits=-zdist/cfg.target_tau
            logits=logits-logits.max()
            p=np.exp(logits)
            p=(p/p.sum()).astype(np.float32)

            feat[row]=f
            base[row]=b
            target[row]=p
            dist[row]=dd
            cand_anchor[row]=mem_a[ix]

    out=dict(
        feat=feat,
        base=base,
        target=target,
        distance=dist,
        candidate_anchor=cand_anchor,
        anchor=np.asarray(query_anchor,np.int64),
        channel=np.asarray(query_channel,np.int64),
    )

    return out

print("[PASS] listwise supervision builder ready")


In [ ]:

# Cell 8 — training / validation functions

def listwise_epoch(model,opt,data,rng):
    model.train()

    n=len(data["feat"])
    order=rng.permutation(n)

    total=0.0
    count=0

    for left in range(0,n,cfg.batch):
        ix=order[left:left+cfg.batch]

        feat=torch.from_numpy(
            data["feat"][ix]
        ).float().to(DEVICE)

        base=torch.from_numpy(
            data["base"][ix]
        ).float().to(DEVICE)

        target=torch.from_numpy(
            data["target"][ix]
        ).float().to(DEVICE)

        score=model(feat,base)
        logp=F.log_softmax(score,dim=1)

        loss=-(target*logp).sum(dim=1).mean()

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            cfg.grad_clip,
        )
        opt.step()

        total+=float(loss.detach())*len(ix)
        count+=len(ix)

    return total/max(count,1)

@torch.no_grad()
def analog_mse(model,data):
    model.eval()

    vals=[]

    for left in range(0,len(data["feat"]),256):
        right=min(len(data["feat"]),left+256)

        feat=torch.from_numpy(
            data["feat"][left:right]
        ).float().to(DEVICE)

        base=torch.from_numpy(
            data["base"][left:right]
        ).float().to(DEVICE)

        score=model(feat,base).cpu().numpy()

        top=np.argsort(
            -score,
            axis=1,
        )[:,:cfg.top_k]

        d=np.take_along_axis(
            data["distance"][left:right],
            top,
            axis=1,
        )

        vals.append(d.mean(axis=1))

    return float(np.concatenate(vals).mean())

def fit_phase_a(train_data,val_data,seed):
    set_seed(seed)

    model=PRRStatReranker().to(DEVICE)
    opt=torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.wd,
    )

    rng=np.random.default_rng(seed+101)

    best=np.inf
    best_epoch=1
    best_state=None
    wait=0
    history=[]

    for epoch in range(1,cfg.max_epochs+1):
        train_loss=listwise_epoch(
            model,opt,train_data,rng
        )

        val_metric=analog_mse(
            model,val_data
        )

        history.append({
            "epoch":epoch,
            "train_fc_loss":train_loss,
            "val_analog_mse":val_metric,
            "alpha":float(
                F.softplus(
                    model.raw_alpha
                ).detach().cpu()
            ),
        })

        print(
            f"  ep={epoch:02d} "
            f"FC={train_loss:.5f} "
            f"valAnalog={val_metric:.6f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if val_metric<best-1e-10:
            best=val_metric
            best_epoch=epoch
            best_state={
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            wait=0
        else:
            wait+=1

        if wait>=cfg.patience:
            break

    model.load_state_dict(best_state,strict=True)
    return model,best_epoch,pd.DataFrame(history)

def refit_phase_b(combined_data,best_epoch,seed):
    set_seed(seed)

    model=PRRStatReranker().to(DEVICE)
    opt=torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.wd,
    )

    rng=np.random.default_rng(seed+202)

    for _ in range(int(best_epoch)):
        listwise_epoch(
            model,opt,combined_data,rng
        )

    model.eval()
    return model

print("[PASS] training code ready")


In [ ]:

# Cell 9 — retrieval forecast from a trained PRR-Stat model

@torch.no_grad()
def retrieval_forecast(
    dataset,
    h,
    query_anchor,
    query_channel,
    memory_boundary,
    med,
    iqr,
    model,
):
    z=DATA[dataset]["z"]
    N=len(query_anchor)
    M=cfg.top_m

    retrieved=np.empty((N,h),np.float32)
    analog=np.empty(N,np.float32)

    stride=memory_stride(dataset)

    for ch in np.unique(query_channel):
        rows=np.flatnonzero(query_channel==ch)
        qa=query_anchor[rows]

        mem_a=np.arange(
            cfg.retrieval_seq_len,
            int(memory_boundary)-h+1,
            stride,
            dtype=np.int64,
        )

        mem_past=past_windows(z,int(ch),mem_a)
        mem_pat=batch_pattern(mem_past)
        mem_stat=scale_stats(
            context7(mem_past),
            med,
            iqr,
        )
        mem_future=future_residuals(
            z,int(ch),mem_a,h
        )

        q_past=past_windows(z,int(ch),qa)
        q_pat=batch_pattern(q_past)
        q_stat=scale_stats(
            context7(q_past),
            med,
            iqr,
        )
        q_future=future_residuals(
            z,int(ch),qa,h
        )

        ps=q_pat@mem_pat.T

        for local,row in enumerate(rows):
            ix=top_m_indices(
                ps[local],
                M,
            )

            b=ps[local,ix]
            cs=mem_stat[ix]
            qs=q_stat[local]

            qrep=np.repeat(
                qs[None,:],
                M,
                axis=0,
            )

            feat=np.concatenate(
                [
                    b[:,None],
                    qrep,
                    cs,
                    qrep-cs,
                    np.abs(qrep-cs),
                ],
                axis=1,
            ).astype(np.float32)

            score=model(
                torch.from_numpy(
                    feat[None,:,:]
                ).float().to(DEVICE),
                torch.from_numpy(
                    b[None,:].astype(np.float32)
                ).float().to(DEVICE),
            ).cpu().numpy()[0]

            top=np.argsort(
                -score
            )[:cfg.top_k]

            chosen=mem_future[ix[top]]

            qcur=z[int(qa[local])-1,int(ch)]

            retrieved[row]=(
                chosen.mean(axis=0)
                +qcur
            )

            analog[row]=np.mean(
                (
                    chosen
                    -q_future[local][None,:]
                )**2
            )

    return retrieved,analog

print("[PASS] retrieval forecast code ready")


In [ ]:

# Cell 10 — run PRR-Stat 24 conditions, resumably

def condition_json(dataset,h):
    return RESULT_ROOT/f"{dataset}_H{h}_result.json"

def condition_npz(dataset,h):
    return CACHE_ROOT/f"{dataset}_H{h}_test.npz"

def condition_done(dataset,h):
    return (
        condition_json(dataset,h).exists()
        and condition_npz(dataset,h).exists()
    )

all_rows=[]

# recover prior completed rows
for dataset in DATASETS:
    for h in HORIZONS:
        if condition_done(dataset,h):
            all_rows.append(
                json.loads(
                    condition_json(dataset,h).read_text()
                )
            )

done_keys={
    (r["dataset"],int(r["horizon"]))
    for r in all_rows
}

for dataset in DATASETS:
    z=DATA[dataset]["z"]
    train_end=DATA[dataset]["train_end"]
    val_end=DATA[dataset]["val_end"]

    for h in HORIZONS:
        if (dataset,h) in done_keys:
            print("[SKIP]",dataset,h)
            continue

        print("\n"+"="*110)
        print("PRR-Stat refit:",dataset,"H=",h)
        print("="*110)

        med,iqr=fit_feature_scaler(
            dataset,h
        )

        initial_boundary=int(
            cfg.initial_memory_fraction
            *train_end
        )

        train_anchor_pool=np.arange(
            max(
                initial_boundary,
                cfg.retrieval_seq_len,
            ),
            train_end-h+1,
            dtype=np.int64,
        )

        cap=(
            cfg.train_cap_mechanism
            if dataset in ("ETTh1","Weather")
            else cfg.train_cap_frozen
        )

        train_a,train_c=balanced_pairs(
            dataset,
            train_anchor_pool,
            cap,
            seed=36000+100*HORIZONS.index(h)+DATASETS.index(dataset),
        )

        val_a,val_c,val_direct,val_truth=load_direct(
            dataset,h,"val"
        )

        # Phase A supervision.
        t0=time.perf_counter()

        train_sup=build_supervision(
            dataset,h,
            train_a,train_c,
            initial_boundary,
            med,iqr,
        )

        val_sup=build_supervision(
            dataset,h,
            val_a,val_c,
            train_end,
            med,iqr,
        )

        print(
            "Supervision built:",
            len(train_a),"train,",
            len(val_a),"val; sec=",
            round(time.perf_counter()-t0,1)
        )

        phase_a,best_epoch,hist=fit_phase_a(
            train_sup,
            val_sup,
            cfg.seed,
        )

        hist.to_csv(
            RESULT_ROOT/f"{dataset}_H{h}_phaseA_history.csv",
            index=False,
        )

        # Validation retrieval from the validation-selected Phase-A model.
        val_retrieval,val_analog=retrieval_forecast(
            dataset,h,
            val_a,val_c,
            train_end,
            med,iqr,
            phase_a,
        )

        # Validation-only global analytic beta for a common integration comparison.
        e=(
            val_direct-val_truth
        ).astype(np.float64)

        delta=(
            val_retrieval-val_direct
        ).astype(np.float64)

        beta_global=float(
            np.clip(
                -np.sum(e*delta)
                /(np.sum(delta*delta)+1e-12),
                0.0,
                1.0,
            )
        )

        # Phase B: refit from scratch using train + validation historical supervision.
        combined={
            k:np.concatenate(
                [train_sup[k],val_sup[k]],
                axis=0,
            )
            for k in (
                "feat","base","target",
                "distance","candidate_anchor",
                "anchor","channel"
            )
        }

        final_model=refit_phase_b(
            combined,
            best_epoch,
            cfg.seed,
        )

        ckpt={
            "Dataset":dataset,
            "Horizon":h,
            "Seed":cfg.seed,
            "InputDim":29,
            "BestEpoch":int(best_epoch),
            "FeatureMedian":med,
            "FeatureIQR":iqr,
            "StateDict":{
                k:v.detach().cpu().clone()
                for k,v in final_model.state_dict().items()
            },
            "Recipe":"PRR-Stat paper recipe",
        }

        torch.save(
            ckpt,
            CKPT_ROOT/f"{dataset}_H{h}_PRRStat_seed0.pt",
        )

        # Test uses training + validation memory.
        test_a,test_c,test_direct,test_truth=load_direct(
            dataset,h,"test"
        )

        test_retrieval,test_analog=retrieval_forecast(
            dataset,h,
            test_a,test_c,
            val_end,
            med,iqr,
            final_model,
        )

        pred_global=(
            test_direct
            +beta_global*(
                test_retrieval-test_direct
            )
        )

        pred_fixed01=(
            test_direct
            +0.1*(
                test_retrieval-test_direct
            )
        )

        def mse(x,y):
            return float(np.mean(
                (
                    x.astype(np.float64)
                    -y.astype(np.float64)
                )**2
            ))

        def mae(x,y):
            return float(np.mean(
                np.abs(
                    x.astype(np.float64)
                    -y.astype(np.float64)
                )
            ))

        dm=mse(test_direct,test_truth)
        gm=mse(pred_global,test_truth)
        fm=mse(pred_fixed01,test_truth)

        row={
            "dataset":dataset,
            "horizon":int(h),
            "seed":cfg.seed,
            "best_epoch":int(best_epoch),
            "beta_global":beta_global,
            "val_analog_future_mse":float(val_analog.mean()),
            "test_analog_future_mse":float(test_analog.mean()),
            "direct_mse":dm,
            "prr_stat_global_mse":gm,
            "prr_stat_fixed01_mse":fm,
            "prr_stat_global_gain_vs_direct_pct":
                100*(dm-gm)/dm,
            "prr_stat_fixed01_gain_vs_direct_pct":
                100*(dm-fm)/dm,
            "direct_mae":mae(test_direct,test_truth),
            "prr_stat_global_mae":mae(pred_global,test_truth),
        }

        np.savez_compressed(
            condition_npz(dataset,h),
            anchor=test_a,
            channel=test_c,
            retrieval=test_retrieval,
            analog_mse=test_analog,
            beta_global=np.asarray(beta_global),
        )

        condition_json(dataset,h).write_text(
            json.dumps(row,indent=2),
            encoding="utf-8",
        )

        all_rows.append(row)

        print(
            f"TEST Direct={dm:.6f} "
            f"PRRStatGlobal={gm:.6f} "
            f"Fixed0.1={fm:.6f} "
            f"beta={beta_global:.4f} "
            f"gain={row['prr_stat_global_gain_vs_direct_pct']:+.3f}%"
        )

        del train_sup,val_sup,combined,phase_a,final_model
        gc.collect()
        torch.cuda.empty_cache()

prr_stat_df=pd.DataFrame(all_rows).drop_duplicates(
    ["dataset","horizon"],
    keep="last",
).sort_values(
    ["dataset","horizon"]
).reset_index(drop=True)

display(prr_stat_df)

assert len(prr_stat_df)==24

prr_stat_df.to_csv(
    RESULT_ROOT/"patchtst24_prr_stat_seed0.csv",
    index=False,
)

print("[PASS] PRR-Stat refit/evaluation = 24/24")


In [ ]:

# Cell 11 — merge with Exp35B PRR results

prr_csv=EXP35/"patchtst_24_prr_results.csv"
assert prr_csv.exists(),prr_csv

prr_df=pd.read_csv(prr_csv)
prr_df["horizon"]=prr_df["horizon"].astype(int)

cmp=prr_stat_df.merge(
    prr_df[
        [
            "dataset","horizon",
            "test_analog_future_mse",
            "global_mse",
            "fixed01_mse",
            "beta_global",
            "global_gain_pct",
        ]
    ].rename(
        columns={
            "test_analog_future_mse":
                "prr_analog_future_mse",
            "global_mse":
                "prr_global_mse",
            "fixed01_mse":
                "prr_fixed01_mse",
            "beta_global":
                "prr_beta_global",
            "global_gain_pct":
                "prr_global_gain_vs_direct_pct",
        }
    ),
    on=["dataset","horizon"],
    how="inner",
)

assert len(cmp)==24

cmp["retrieval_gain_prr_vs_prr_stat_pct"]=(
    100
    *(
        cmp.test_analog_future_mse
        -cmp.prr_analog_future_mse
    )
    /cmp.test_analog_future_mse
)

cmp["forecast_global_gain_prr_vs_prr_stat_pct"]=(
    100
    *(
        cmp.prr_stat_global_mse
        -cmp.prr_global_mse
    )
    /cmp.prr_stat_global_mse
)

cmp["forecast_fixed01_gain_prr_vs_prr_stat_pct"]=(
    100
    *(
        cmp.prr_stat_fixed01_mse
        -cmp.prr_fixed01_mse
    )
    /cmp.prr_stat_fixed01_mse
)

display(
    cmp[
        [
            "dataset","horizon",
            "direct_mse",
            "prr_stat_global_mse",
            "prr_global_mse",
            "prr_stat_fixed01_mse",
            "prr_fixed01_mse",
            "test_analog_future_mse",
            "prr_analog_future_mse",
            "retrieval_gain_prr_vs_prr_stat_pct",
            "forecast_global_gain_prr_vs_prr_stat_pct",
            "forecast_fixed01_gain_prr_vs_prr_stat_pct",
            "beta_global",
            "prr_beta_global",
        ]
    ]
)

cmp.to_csv(
    RESULT_ROOT/"patchtst24_prr_stat_vs_prr.csv",
    index=False,
)


In [ ]:

# Cell 12 — final comparison summary

retrieval_win=(
    cmp.prr_analog_future_mse
    <cmp.test_analog_future_mse
)

global_win=(
    cmp.prr_global_mse
    <cmp.prr_stat_global_mse
)

fixed_win=(
    cmp.prr_fixed01_mse
    <cmp.prr_stat_fixed01_mse
)

print("="*105)
print("PRR vs PRR-Stat — PatchTST × 24")
print("="*105)

print(
    f"Retrieval wins                 : {int(retrieval_win.sum())}/24"
)

print(
    f"Forecast wins, GlobalAnalytic : {int(global_win.sum())}/24"
)

print(
    f"Forecast wins, fixed beta=.1 : {int(fixed_win.sum())}/24"
)

print(
    "Mean retrieval gain vs PRR-Stat:",
    f"{cmp.retrieval_gain_prr_vs_prr_stat_pct.mean():+.3f}%"
)

print(
    "Mean forecast gain vs PRR-Stat (Global diagnostic):",
    f"{cmp.forecast_global_gain_prr_vs_prr_stat_pct.mean():+.3f}%"
)

print(
    "Median forecast gain vs PRR-Stat (Global diagnostic):",
    f"{cmp.forecast_global_gain_prr_vs_prr_stat_pct.median():+.3f}%"
)

print(
    "Mean forecast gain vs PRR-Stat (fixed beta=.1):",
    f"{cmp.forecast_fixed01_gain_prr_vs_prr_stat_pct.mean():+.3f}%"
)

dataset_summary=(
    cmp.assign(
        retrieval_win=retrieval_win,
        global_win=global_win,
        fixed_win=fixed_win,
    )
    .groupby("dataset",sort=False)
    .agg(
        retrieval_wins=("retrieval_win","sum"),
        global_forecast_wins=("global_win","sum"),
        fixed01_forecast_wins=("fixed_win","sum"),
        mean_retrieval_gain_pct=(
            "retrieval_gain_prr_vs_prr_stat_pct","mean"
        ),
        mean_global_forecast_gain_pct=(
            "forecast_global_gain_prr_vs_prr_stat_pct","mean"
        ),
        mean_fixed01_forecast_gain_pct=(
            "forecast_fixed01_gain_prr_vs_prr_stat_pct","mean"
        ),
    )
    .reset_index()
)

display(dataset_summary)

dataset_summary.to_csv(
    RESULT_ROOT/"dataset_summary_prr_stat_vs_prr.csv",
    index=False,
)

summary={
    "experiment":"Exp36R",
    "comparison":"PRR-Stat vs PRR",
    "backbone":"PatchTST",
    "conditions":24,
    "seed":0,
    "prr_stat_refit":True,
    "forecaster_forward_performed":False,
    "retrieval_wins_prr_vs_prr_stat":
        int(retrieval_win.sum()),
    "global_forecast_wins_prr_vs_prr_stat":
        int(global_win.sum()),
    "fixed01_forecast_wins_prr_vs_prr_stat":
        int(fixed_win.sum()),
    "mean_retrieval_gain_pct":
        float(cmp.retrieval_gain_prr_vs_prr_stat_pct.mean()),
    "mean_global_forecast_gain_pct":
        float(cmp.forecast_global_gain_prr_vs_prr_stat_pct.mean()),
    "median_global_forecast_gain_pct":
        float(cmp.forecast_global_gain_prr_vs_prr_stat_pct.median()),
    "mean_fixed01_forecast_gain_pct":
        float(cmp.forecast_fixed01_gain_prr_vs_prr_stat_pct.mean()),
}

(RESULT_ROOT/"exp36R_summary.json").write_text(
    json.dumps(summary,indent=2),
    encoding="utf-8",
)

print(json.dumps(summary,indent=2))

print("\nEXP36R COMPLETE")
print("Next: expand Direct / PRR-Stat / PRR to the remaining four forecasters.")
